# Entrenamiento con el dataset de Duotek (Colab)

Entrena `exp_00_base` (YOLOv8n) y `exp_04_yolo26` (YOLO26n) con el reparto por secuencias
(`scripts/03_repartir_secuencias.py`) y compara ambos en `logs/comparacion.csv`.

**Antes de empezar** (Runtime → Change runtime type → GPU, por ejemplo T4):
1. En Drive deja la carpeta del repo (con `src/`, `configs/`, `scripts/`, `uv.lock`) y el `.tar` del dataset.
2. Ajusta las dos rutas de la siguiente celda.

El dataset se copia al disco local de Colab (`/content`): entrenar leyendo directo desde Drive es muy lento.
Pesos y logs se guardan en Drive, así sobreviven si la sesión se desconecta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = "/content/drive/MyDrive/tracker-duotek"            # carpeta del repo en Drive
TAR  = "/content/drive/MyDrive/duotek/dataset.tar"        # dataset descargado (.tar)
RUNS = "/content/drive/MyDrive/tracker-duotek-runs"       # corridas completas (best.pt, curvas)

%cd "$REPO"
!ls

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU: cambia el tipo de entorno")

In [ ]:
# Misma versión de ultralytics que fija uv.lock, para que Colab y el repo entrenen igual.
import re, subprocess, sys

txt = open("uv.lock").read()
m = re.search(r'name = "ultralytics"\nversion = "([^"]+)"', txt)
version = m.group(1) if m else None
print("Versión en uv.lock:", version)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"ultralytics=={version}" if version else "ultralytics", "pyyaml"], check=True)

## 1. Dataset: copiar, descomprimir, repartir y validar

In [ ]:
!mkdir -p /content/dataset
!cp "$TAR" /content/dataset.tar
!tar -xf /content/dataset.tar -C /content/dataset
!find /content/dataset -maxdepth 3 -type d | head -20

In [ ]:
# Reparto por secuencia completa (val: 8,15,21,33,35,55,59 / test: 10,29,40,47,52,60 / resto: train).
!python scripts/03_repartir_secuencias.py --datasets /content/dataset --limpiar
!python scripts/04_validar_reparticion.py --datasets /content/dataset

In [ ]:
# Debe decir: TRAIN 36 sec / 12077 imágenes, VAL 7 / 2618, TEST 6 / 2507 y TODO OK.
DATA = "/content/dataset/dataset_segmentado/dataset_yolo/data.yaml"
print(open(DATA).read())

## 2. Prueba rápida (1 época) para confirmar que todo corre

In [ ]:
import os
os.environ["SALMONES_RUNS"] = RUNS
!python -m src.entrenar --config configs/exp_00_base.yaml --data "$DATA" --epochs 1 --nombre prueba_1ep

In [ ]:
!rm -f weights/prueba_1ep.pt logs/prueba_1ep.json logs/prueba_1ep_epocas.csv logs/prueba_1ep_val.json
!rm -rf "$RUNS/prueba_1ep"
# La prueba dejó una fila en logs/comparacion.csv: bórrala para que la tabla final solo tenga las corridas reales.
!sed -i '/^prueba_1ep,/d' logs/comparacion.csv

## 3. Experimento 0: línea base (YOLOv8n)

In [ ]:
!python -m src.entrenar --config configs/exp_00_base.yaml --data "$DATA"

## 4. Experimento 4: YOLO26n (único cambio: el modelo)

In [ ]:
!python -m src.entrenar --config configs/exp_04_yolo26.yaml --data "$DATA"

## 5. Evaluación en test (solo cuando ya elegiste el modelo)

Test no se usa para decidir nada durante el entrenamiento; se mira una vez al final.

In [ ]:
!python -m src.entrenar --evaluar weights/exp_00_base.pt --data "$DATA" --split test
!python -m src.entrenar --evaluar weights/exp_04_yolo26.pt --data "$DATA" --split test

In [ ]:
import pandas as pd
pd.read_csv("logs/comparacion.csv")

Los pesos quedan en `weights/<nombre>.pt` (dentro del repo en Drive). Descárgalos y úsalos con
`python -m src.comparar_seguidores --pesos weights/<nombre>.pt ...` para ver el error de conteo con el modelo nuevo.